# Maia Lite 0.3.0 — A100 Pretraining Engine SSD v2.3

Preserves v2.2 and adds full-model A100 BF16 checkpoint/resume validation with synthetic data.


In [1]:
from pathlib import Path
import os, json, math, time, random, hashlib, bisect, gc, datetime, shutil
from dataclasses import asdict
from collections import defaultdict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
except ImportError:
    print('Colab Drive not available; local validation only')
PROJECT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
SSD=Path('/content/maia_v8_ssd/dataset')
MANIFEST=SSD/'dataset_manifest.json'
APPROVALS=PROJECT/'corpus_v8/approvals/shard_eligibility_v1.jsonl'
PRODUCTION=False  # Never switch on until independent rights review and preflight pass
SEED=20261105
print('Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
print('SSD manifest present:',MANIFEST.is_file())

Mounted at /content/drive
Torch: 2.14.1+cu130 CUDA: True
SSD manifest present: False


## Bootstrap automático do corpus v8 (CPU)

Execute a célula seguinte **antes** da célula `assert MANIFEST.is_file()`.

- Se o dataset SSD estiver completo, será reutilizado.
- Se não existir, reconstrói os 18 binários a partir dos 28 ZIPs.
- Confere SHA-256 dos ZIPs e de cada shard, além de contagens e índices.
- Não lê shards remotos individuais, não executa treinamento e não altera o Drive.

A preparação anterior levou 15,6 minutos; o tempo depende do runtime e do Drive.

In [2]:
# CELL 3 — AUTO-BOOTSTRAP: run once after configuration, before corpus integrity gate.
import sqlite3, zipfile
from collections import defaultdict

CORPUS_ROOT = PROJECT / 'corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full'
PACK_DIR = CORPUS_ROOT / 'packed_zip_v1'
DB_PATH = CORPUS_ROOT / 'recovery_audit_v4/state_operational_v5.sqlite3'
SSD_PARENT = SSD.parent
WORK = SSD_PARENT / 'dataset.partial'
ZIP_TMP = SSD_PARENT / 'current_pack.zip'
BLOCK_BYTES = 2048 * 4
PACK_LIMIT = 1024**3

def _hash_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(4 * 1024**2), b''):
            h.update(b)
    return h.hexdigest()

def _check_local_dataset():
    if not MANIFEST.is_file():
        return False
    try:
        m = json.loads(MANIFEST.read_text())
        if not (m.get('status') == 'complete' and m.get('total_blocks') == 3607119
                and m.get('total_shards') == 60612 and m.get('tokens_per_block') == 2048
                and m.get('dtype') == '<u4' and len(m.get('splits', [])) == 18):
            return False
        for e in m['splits']:
            f = SSD / e['relative_path']
            idx = SSD / e['index_path']
            if not f.is_file() or f.stat().st_size != e['blocks'] * BLOCK_BYTES or not idx.is_file():
                return False
        return True
    except (OSError, ValueError, KeyError):
        return False

def _build_local_dataset():
    if _check_local_dataset():
        print('REUSE: SSD dataset already complete:', SSD)
        return
    if SSD.exists():
        raise RuntimeError('Existing SSD dataset is incomplete; inspect/remove it manually: ' + str(SSD))
    assert PACK_DIR.is_dir(), f'ZIP pack directory missing: {PACK_DIR}'
    assert DB_PATH.is_file(), f'SQLite catalog missing: {DB_PATH}'
    SSD_PARENT.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(DB_PATH.as_uri() + '?mode=ro', uri=True, timeout=60)
    try:
        rows = conn.execute('SELECT source, split, shard_index, path, blocks, sha256 FROM shard ORDER BY source, split, shard_index').fetchall()
    finally:
        conn.close()
    assert len(rows) == 60612 and sum(r[4] for r in rows) == 3607119
    packs, group, size = [], [], 0
    for r in rows:
        nbytes = r[4] * BLOCK_BYTES
        if group and size + nbytes > PACK_LIMIT:
            packs.append(group)
            group, size = [], 0
        group.append(r)
        size += nbytes
    if group:
        packs.append(group)
    assert len(packs) == 28
    required = sum(r[4] for r in rows) * BLOCK_BYTES
    free = shutil.disk_usage(SSD_PARENT).free
    print(f'Corpus: {required/1024**3:.2f} GiB | free: {free/1024**3:.2f} GiB')
    assert free >= required + 2*1024**3, 'Insufficient SSD space'
    if WORK.exists():
        shutil.rmtree(WORK)  # interrupted build only; never touches completed dataset
    WORK.mkdir(parents=True)
    handles, indices = {}, {}
    counts, shards = defaultdict(int), defaultdict(int)
    start = time.monotonic()
    try:
        for i, pack in enumerate(packs):
            src, split = pack[0][:2]
            name = f'pack_{i:04d}_{src}_{split}.zip'
            meta = json.loads((PACK_DIR / (name + '.json')).read_text())
            assert meta['status'] == 'complete' and meta['zip_name'] == name and meta['count'] == len(pack)
            expected = {f'{r[0]}/{r[1]}/{r[0]}_{r[1]}_{r[2]:06d}.bin':r for r in pack}
            ZIP_TMP.unlink(missing_ok=True)
            print(f'[{i+1}/28] {name}', flush=True)
            shutil.copyfile(PACK_DIR / name, ZIP_TMP)
            assert ZIP_TMP.stat().st_size == meta['zip_bytes']
            assert _hash_file(ZIP_TMP) == meta['zip_sha256']
            with zipfile.ZipFile(ZIP_TMP) as z:
                assert len(z.infolist()) == len(pack)
                seen = set()
                for member in z.infolist():
                    arc = member.filename
                    assert arc in expected and arc not in seen
                    assert member.compress_type == zipfile.ZIP_STORED
                    seen.add(arc)
                    source, split, shard_idx, _, blocks, sha = expected[arc]
                    assert member.file_size == blocks * BLOCK_BYTES
                    key = (source, split)
                    folder = WORK / source
                    folder.mkdir(exist_ok=True)
                    if key not in handles:
                        handles[key] = open(folder / f'{split}.bin', 'wb')
                        indices[key] = open(folder / f'{split}.index.jsonl', 'w', encoding='utf-8')
                    digest = hashlib.sha256()
                    written = 0
                    with z.open(member) as f:
                        while True:
                            buf = f.read(4 * 1024**2)
                            if not buf:
                                break
                            handles[key].write(buf)
                            digest.update(buf)
                            written += len(buf)
                    assert written == blocks * BLOCK_BYTES and digest.hexdigest() == sha.lower(), f'Shard checksum mismatch: {arc}'
                    indices[key].write(json.dumps({'shard_index': shard_idx, 'start_block': counts[key], 'blocks': blocks, 'sha256': sha}) + '\n')
                    counts[key] += blocks
                    shards[key] += 1
                assert seen == set(expected)
            ZIP_TMP.unlink()
            print(f'  OK | {i+1}/28 | {(time.monotonic()-start)/60:.1f} min', flush=True)
        for f in [*handles.values(), *indices.values()]:
            f.flush()
            os.fsync(f.fileno())
            f.close()
        handles.clear(); indices.clear()
        assert sum(counts.values()) == 3607119 and sum(shards.values()) == 60612
        manifest = {'status': 'complete', 'tokens_per_block': 2048, 'dtype': '<u4',
                    'total_blocks': 3607119, 'total_shards': 60612, 'splits': []}
        for (source, split), blocks in sorted(counts.items()):
            f = WORK / source / f'{split}.bin'
            assert f.stat().st_size == blocks * BLOCK_BYTES
            manifest['splits'].append({'source': source, 'split': split, 'blocks': blocks,
              'shards': shards[(source,split)], 'relative_path': f'{source}/{split}.bin',
              'index_path': f'{source}/{split}.index.jsonl'})
        assert len(manifest['splits']) == 18
        (WORK / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2))
        WORK.rename(SSD)
        print('PASS: SSD DATASET PREPARED', flush=True)
    finally:
        for f in [*handles.values(), *indices.values()]:
            if not f.closed:
                f.close()
        ZIP_TMP.unlink(missing_ok=True)

_build_local_dataset()
assert _check_local_dataset(), 'SSD-first bootstrap failed validation'
print('PASS: SSD-FIRST DATASET READY')


Corpus: 27.52 GiB | free: 235.26 GiB
[1/28] pack_0000_fineweb2_es_test.zip
  OK | 1/28 | 0.4 min
[2/28] pack_0001_fineweb2_es_train.zip
  OK | 2/28 | 0.8 min
[3/28] pack_0002_fineweb2_es_train.zip
  OK | 3/28 | 1.2 min
[4/28] pack_0003_fineweb2_es_train.zip
  OK | 4/28 | 1.6 min
[5/28] pack_0004_fineweb2_es_train.zip
  OK | 5/28 | 1.9 min
[6/28] pack_0005_fineweb2_es_train.zip
  OK | 6/28 | 2.3 min
[7/28] pack_0006_fineweb2_pt_train.zip
  OK | 7/28 | 2.7 min
[8/28] pack_0007_fineweb2_pt_train.zip
  OK | 8/28 | 3.0 min
[9/28] pack_0008_fineweb2_pt_train.zip
  OK | 9/28 | 3.4 min
[10/28] pack_0009_fineweb2_pt_train.zip
  OK | 10/28 | 3.7 min
[11/28] pack_0010_fineweb2_pt_train.zip
  OK | 11/28 | 4.1 min
[12/28] pack_0011_fineweb2_pt_train.zip
  OK | 12/28 | 4.4 min
[13/28] pack_0012_fineweb2_pt_train.zip
  OK | 13/28 | 4.8 min
[14/28] pack_0013_fineweb2_pt_train.zip
  OK | 14/28 | 5.3 min
[15/28] pack_0014_fineweb2_pt_train.zip
  OK | 15/28 | 5.6 min
[16/28] pack_0015_fineweb_edu_en_trai

## Model architecture — copied unchanged from Synthetic Engine v1

In [3]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss
model=MaiaGPT(cfg)
params=sum(p.numel() for p in model.parameters())
print('Parameters:',format(params,','),'(~%.3f B)'%(params/1e9))
assert 480_000_000<=params<=530_000_000
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()

Parameters: 507,026,432 (~0.507 B)


## v2.3 — A100 BF16 full-model checkpoint/resume validation

**Fast path:** execute only cells **1 (imports/config)**, **5 (original architecture)**, and the **A100 validation code immediately below**. This does **not** require SSD consolidation, licensing approval, or real data. The other v2.2 cells are retained for the original pipeline and CPU test.

- Full original 507M model; FP32 master weights, BF16 autocast, AdamW, gradient accumulation, gradient clipping and a simple deterministic step-dependent learning rate.
- Synthetic batches addressed by `(seed, step, microstep)`; causal next-token targets.
- Compare **2 continuous updates** against **1 update → atomic checkpoint → restore → 1 update**. Check weights, optimizer, RNG, loss, batch hashes and learning-rate sequence.
- GPU checkpoint written to **ephemeral SSD**, not Drive, with SHA-256; model and optimizer restored from disk. This is a validation checkpoint, **not** a production checkpoint.
- Explicit SDPA math backend and deterministic algorithms; a backend determinism error fails the test rather than being silently ignored. **Exact equality is a test goal, not a guarantee.**
- A100 required. This test allocates full-model optimizer state and runs both paths sequentially. It can take minutes and several GiB of GPU RAM. `CUDA_VISIBLE_DEVICES` should be configured before runtime startup if needed.
- `PRODUCTION=False` and `LICENSE_GATE` are never changed. No corpus tokens are read.


In [ ]:
# v2.3 FULL-MODEL A100 PREFLIGHT — independent of corpus / approval / SSD dataset
import contextlib, tempfile, copy, pickle
RUN_A100_VALIDATION = True
A100_TEST_SEED = 20261107
A100_TEST_STEPS = 2
A100_TEST_SPLIT = 1
A100_TEST_ACCUM = 2
A100_TEST_MICROBATCH = 1
A100_TEST_SEQ = 64  # intentionally short for fast full-parameter validation
A100_TEST_LR = 1e-4
A100_TEST_WD = 0.1

if RUN_A100_VALIDATION:
    assert torch.cuda.is_available(), 'A100 validation requires CUDA'
    assert PRODUCTION is False, 'This validation must not enable production'
    assert A100_TEST_SEQ + 1 <= cfg.block_size
    assert sum(p.numel() for p in model.parameters()) == 507026432
    # CUBLAS_WORKSPACE_CONFIG must be set before the first CUDA context is initialized.
    # If this is too late, restart runtime and rerun this fast path from the top.
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
    try:
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
    except AttributeError:
        pass
    gpu_name = torch.cuda.get_device_name(0)
    assert 'A100' in gpu_name.upper(), f'Expected A100, got {gpu_name}'
    print('A100 FULL-MODEL VALIDATION:', gpu_name, '| torch', torch.__version__)
    del model
    gc.collect(); torch.cuda.empty_cache()

    def gpu_seed(seed):
        random.seed(seed); np.random.seed(seed)
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    def gpu_batch(step, micro):
        digest = hashlib.sha256(f'maia-v23:{A100_TEST_SEED}:{step}:{micro}'.encode()).digest()
        rng = np.random.default_rng(int.from_bytes(digest[:8], 'little'))
        raw = rng.integers(0, cfg.vocab_size,
            size=(A100_TEST_MICROBATCH, A100_TEST_SEQ + 1), dtype=np.int64)
        h = hashlib.sha256(raw.tobytes()).hexdigest()
        return (torch.from_numpy(raw[:, :-1].copy()).cuda(),
                torch.from_numpy(raw[:, 1:].copy()).cuda(), h)

    def new_gpu_run():
        gpu_seed(A100_TEST_SEED)
        m = MaiaGPT(cfg).cuda().train()
        opt = torch.optim.AdamW(m.parameters(), lr=A100_TEST_LR,
             betas=(0.9, 0.95), eps=1e-8, weight_decay=A100_TEST_WD, foreach=False)
        return m, opt

    def gpu_updates(m, opt, start, stop, logs):
        for step in range(start, stop):
            tic = time.perf_counter()
            opt.param_groups[0]['lr'] = A100_TEST_LR * (step + 1) / A100_TEST_STEPS
            opt.zero_grad(set_to_none=True)
            hashes=[]; losses=[]
            # Force the math SDPA implementation, rather than nondeterministic fused kernels.
            with torch.nn.attention.sdpa_kernel([torch.nn.attention.SDPBackend.MATH]):
                for micro in range(A100_TEST_ACCUM):
                    x,y,h = gpu_batch(step,micro)
                    hashes.append(h)
                    with torch.autocast('cuda',dtype=torch.bfloat16):
                        _, loss = m(x,y)
                    losses.append(float(loss.detach().cpu()))
                    (loss / A100_TEST_ACCUM).backward()
                    del x,y,loss
            norm = torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
            assert all(math.isfinite(v) for v in losses) and math.isfinite(float(norm))
            opt.step()
            torch.cuda.synchronize()
            row={'step':step+1,'losses':losses,'batch_hashes':hashes,
                 'lr':opt.param_groups[0]['lr'],'seconds':time.perf_counter()-tic}
            logs.append(row)
            print('  update',step+1,'loss',round(sum(losses)/len(losses),6),
                  'seconds',round(row['seconds'],2),
                  'peak_GiB',round(torch.cuda.max_memory_allocated()/2**30,2))

    def cpu_clone(x):
        if torch.is_tensor(x):return x.detach().cpu().clone()
        if isinstance(x,dict):return {k:cpu_clone(v) for k,v in x.items()}
        if isinstance(x,list):return [cpu_clone(v) for v in x]
        if isinstance(x,tuple):return tuple(cpu_clone(v) for v in x)
        return copy.deepcopy(x)

    def gpu_snapshot(m,opt):
        return {'model':cpu_clone(m.state_dict()),
                'optimizer':cpu_clone(opt.state_dict()),
                'python_rng':random.getstate(),
                'numpy_rng':np.random.get_state(),
                'torch_rng':torch.get_rng_state().clone(),
                'cuda_rng': [v.cpu().clone() for v in torch.cuda.get_rng_state_all()]}

    def state_equal(a,b):
        if torch.is_tensor(a) and torch.is_tensor(b):
            return a.dtype==b.dtype and a.shape==b.shape and torch.equal(a,b)
        if isinstance(a,np.ndarray) and isinstance(b,np.ndarray):return np.array_equal(a,b)
        if isinstance(a,dict) and isinstance(b,dict):
            return a.keys()==b.keys() and all(state_equal(a[k],b[k]) for k in a)
        if isinstance(a,(list,tuple)) and isinstance(b,(list,tuple)):
            return len(a)==len(b) and all(state_equal(x,y) for x,y in zip(a,b))
        return a==b

    def file_sha256(path):
        h=hashlib.sha256()
        with open(path,'rb') as f:
            for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
        return h.hexdigest()

    def checkpoint_write(path,m,opt,step):
        state=gpu_snapshot(m,opt)
        state.update({'format':'maia-v23-a100-bf16-preflight',
                      'cfg':asdict(cfg),'seed':A100_TEST_SEED,'step':step,
                      'torch_version':torch.__version__})
        tmp=path.with_suffix('.tmp')
        torch.save(state,tmp)
        os.replace(tmp,path)
        sha=file_sha256(path)
        print('  checkpoint',round(path.stat().st_size/2**30,2),'GiB; sha256:',sha)
        assert file_sha256(path)==sha
        return sha

    def checkpoint_restore(path):
        state=torch.load(path,map_location='cpu',weights_only=False)
        assert state['format']=='maia-v23-a100-bf16-preflight'
        assert state['cfg']==asdict(cfg) and state['seed']==A100_TEST_SEED
        m,opt=new_gpu_run()
        m.load_state_dict(state['model'])
        opt.load_state_dict(state['optimizer'])
        # AdamW may legitimately keep scalar 'step' counters on CPU.
        # Only the parameter-shaped momentum buffers must be on CUDA.
        assert len(opt.state) > 0, 'AdamW state missing after restore'
        for param, st in opt.state.items():
            for key in ('exp_avg', 'exp_avg_sq'):
                assert key in st, f'Missing AdamW {key} after restore'
                value = st[key]
                assert torch.is_tensor(value), f'AdamW {key} is not a tensor'
                assert value.device == param.device and value.shape == param.shape, (key, value.device, param.device)
            if 'step' in st:
                assert torch.is_tensor(st['step']) and st['step'].numel() == 1
        random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng'])
        torch.cuda.set_rng_state_all(state['cuda_rng'])
        step=int(state['step'])
        del state;gc.collect()
        return m,opt,step

    try:
        torch.cuda.reset_peak_memory_stats()
        print('PATH A: continuous')
        a_log=[]; a_m,a_opt=new_gpu_run()
        gpu_updates(a_m,a_opt,0,A100_TEST_STEPS,a_log)
        a_state=gpu_snapshot(a_m,a_opt)
        del a_m,a_opt;gc.collect();torch.cuda.empty_cache()
        print('PATH B: checkpoint + restore')
        b_log=[];b_m,b_opt=new_gpu_run()
        gpu_updates(b_m,b_opt,0,A100_TEST_SPLIT,b_log)
        with tempfile.TemporaryDirectory(prefix='maia_v23_a100_',dir='/content') as tmpdir:
            ck=Path(tmpdir)/'checkpoint.pt'
            ck_sha=checkpoint_write(ck,b_m,b_opt,A100_TEST_SPLIT)
            del b_m,b_opt;gc.collect();torch.cuda.empty_cache()
            assert file_sha256(ck)==ck_sha,'Checkpoint changed after save'
            b_m,b_opt,restored=checkpoint_restore(ck)
            assert restored==A100_TEST_SPLIT
            gpu_updates(b_m,b_opt,restored,A100_TEST_STEPS,b_log)
        b_state=gpu_snapshot(b_m,b_opt)
        del b_m,b_opt;gc.collect();torch.cuda.empty_cache()
        checks={
            'batch_sequence': [r['batch_hashes'] for r in a_log]==[r['batch_hashes'] for r in b_log],
            'losses_exact': [r['losses'] for r in a_log]==[r['losses'] for r in b_log],
            'lr_sequence': [r['lr'] for r in a_log]==[r['lr'] for r in b_log],
            'model_exact': state_equal(a_state['model'],b_state['model']),
            'adamw_exact': state_equal(a_state['optimizer'],b_state['optimizer']),
            'python_rng_exact': state_equal(a_state['python_rng'],b_state['python_rng']),
            'numpy_rng_exact': state_equal(a_state['numpy_rng'],b_state['numpy_rng']),
            'torch_rng_exact': state_equal(a_state['torch_rng'],b_state['torch_rng']),
            'cuda_rng_exact': state_equal(a_state['cuda_rng'],b_state['cuda_rng']),
        }
        for k,v in checks.items():print(('PASS' if v else 'FAIL')+': '+k)
        assert all(checks.values()),'FAIL: A100 full-model checkpoint/resume differs'
        print('PASS: FULL-MODEL A100 BF16 RESUME — 2 vs 1+1 updates')
        print('Peak GPU allocated GiB:',round(torch.cuda.max_memory_allocated()/2**30,2))
        print('No corpus data used; PRODUCTION remains False; LICENSE_GATE unchanged.')
        del a_state,b_state;gc.collect()
    finally:
        torch.use_deterministic_algorithms(False)


## Corpus integrity and explicit licensing gate

`shard_eligibility_v1.jsonl` is a **separately reviewed** input, not generated from the corpus. Each line must have `source`, `split`, `shard_index`, `sha256`, `eligible_for_training` (JSON boolean), `rights_status`, `reviewer`, `evidence`. Every train shard must be accounted for; unapproved shards are excluded, unknown shards are not silently approved. For production, the file must have at least one eligible shard and every eligible shard must have `rights_status="approved"`, a reviewer and evidence. The original v1 provenance report has pending review: it is not an approval.

In [ ]:
assert MANIFEST.is_file(), 'Run SSD-first consolidation before this notebook'
info=json.loads(MANIFEST.read_text())
assert info['status']=='complete' and info['total_blocks']==3607119 and info['total_shards']==60612
assert info['tokens_per_block']==2048 and info['dtype']=='<u4'
entries={(e['source'],e['split']):e for e in info['splits']}
assert len(entries)==18
for e in info['splits']:
    f=SSD/e['relative_path']
    assert f.is_file() and f.stat().st_size==e['blocks']*2048*4, f
print('SSD structure PASS:',len(entries),'source/split files')

def load_approved_shards():
    if not APPROVALS.is_file():
        print('APPROVAL FILE MISSING: all real training is disabled:',APPROVALS)
        return {}, None
    digest=hashlib.sha256(APPROVALS.read_bytes()).hexdigest()
    rows={}
    with APPROVALS.open() as f:
        for lineno,line in enumerate(f,1):
            if not line.strip(): continue
            r=json.loads(line)
            k=(r['source'],r['split'],int(r['shard_index']))
            assert k not in rows, ('duplicate approval',k)
            assert type(r['eligible_for_training']) is bool, (lineno,'boolean required')
            if r['eligible_for_training']:
                assert r['split']=='train', 'validation/test cannot be approved for training'
                assert r.get('rights_status')=='approved', (k,'rights not approved')
                assert r.get('reviewer') and r.get('evidence'), (k,'review/evidence required')
            rows[k]=r
    return rows,digest

approvals,approval_sha=load_approved_shards()

In [ ]:
# Build index of reviewed train shards; validate each against the consolidated shard index.
# Fail closed: missing approval means EXCLUDED, not inferred approved.
train_ranges=defaultdict(list)
counts=defaultdict(int)
missing=0; total_shards=0
for (source,split),e in sorted(entries.items()):
    if split!='train': continue
    index=SSD/e['index_path']
    assert index.is_file()
    seen=set(); prev_end=0
    with index.open() as f:
        for line in f:
            r=json.loads(line); idx=int(r['shard_index'])
            assert idx not in seen;seen.add(idx)
            start=int(r['start_block']);blocks=int(r['blocks'])
            assert start==prev_end and blocks>0
            prev_end=start+blocks;total_shards+=1
            k=(source,split,idx)
            a=approvals.get(k)
            if a is None: missing+=1
            elif a['eligible_for_training']:
                assert a['sha256'].lower()==r['sha256'].lower(), ('SHA mismatch',k)
                train_ranges[source].append((start,blocks))
                counts[source]+=blocks
    assert prev_end==e['blocks'] and len(seen)==e['shards']
# Reject extraneous or stale approvals, including nonexistent shard identifiers.
all_index_keys=set()
for (source,split),e in entries.items():
    with (SSD/e['index_path']).open() as f:
        for line in f:
            r=json.loads(line)
            k=(source,split,int(r['shard_index']))
            all_index_keys.add(k)
            if k in approvals:
                assert approvals[k]['sha256'].lower()==r['sha256'].lower(), ('SHA mismatch',k)
assert set(approvals).issubset(all_index_keys), 'Unknown approval entries'
print('Train shards examined:',total_shards,'without review record:',missing)
print('Approved train blocks by source:',dict(counts))
print('Approved tokens:',sum(counts.values())*2048)
LICENSE_GATE=bool(approval_sha and sum(counts.values())>0)
print('LICENSE_GATE:',LICENSE_GATE)

## Stateless, checkpoint-reproducible sampler

Sampler maps `(seed, update, microstep, sample position)` to a source and block. A fixed global seed and global update index restore sample order **without a mutable DataLoader shuffle queue**. Source probabilities are explicit. For validation, only non-training splits are used, and a separate reviewed validation policy must be established before reporting meaningful validation metrics.

In [ ]:
# Example source weights, not an assertion of rights or a finalized scientific mixture.
SOURCE_WEIGHTS={'fineweb2_pt':0.30,'wikipedia_pt':0.10,
                'fineweb_edu_en':0.25,'wikipedia_en':0.10,
                'fineweb2_es':0.20,'wikipedia_es':0.05}
class IndexedCorpus:
    def __init__(self,root,ranges,weights):
        self.root=Path(root);self.ranges={};self.cumulative={};self.mmaps={}
        for source,rr in ranges.items():
            if not rr:continue
            self.ranges[source]=rr
            self.cumulative[source]=np.cumsum([x[1] for x in rr],dtype=np.int64)
            e=entries[(source,'train')]
            self.mmaps[source]=np.memmap(self.root/e['relative_path'],dtype='<u4',mode='r',shape=(e['blocks'],2048))
        self.sources=[s for s in sorted(self.mmaps) if weights.get(s,0)>0]
        if self.sources:
            w=np.array([weights[s] for s in self.sources],dtype=np.float64)
            self.probs=w/w.sum()
    def block(self,source,ordinal):
        ends=self.cumulative[source]
        j=int(np.searchsorted(ends,ordinal,side='right'))
        before=0 if j==0 else int(ends[j-1])
        start,length=self.ranges[source][j]
        ids=np.asarray(self.mmaps[source][start+ordinal-before],dtype=np.int64)
        assert ids.shape==(2048,) and ids.min()>=0 and ids.max()<cfg.vocab_size
        return ids
    def batch(self,update,micro,batch_size,seed=SEED):
        # Stateless seed derivation; no dependence on DataLoader worker scheduling.
        h=hashlib.sha256(f'{seed}:{update}:{micro}'.encode()).digest()
        rng=np.random.default_rng(int.from_bytes(h[:8],'little'))
        chosen=rng.choice(len(self.sources),size=batch_size,p=self.probs)
        x=np.empty((batch_size,2047),dtype=np.int64)
        y=np.empty_like(x)
        for i,si in enumerate(chosen):
            s=self.sources[int(si)]
            ordinal=int(rng.integers(0,int(self.cumulative[s][-1])))
            tokens=self.block(s,ordinal)
            x[i]=tokens[:-1];y[i]=tokens[1:]
        return torch.from_numpy(x),torch.from_numpy(y)

corpus=IndexedCorpus(SSD,train_ranges,SOURCE_WEIGHTS) if LICENSE_GATE else None
if corpus:
    x,y=corpus.batch(0,0,2)
    xx,yy=corpus.batch(0,0,2)
    assert torch.equal(x,xx) and torch.equal(y,yy)
    assert torch.equal(x[:,1:],y[:,:-1])
    print('SAMPLER PASS:',x.shape,'sources:',corpus.sources)
else:
    print('SAMPLER BLOCKED: missing approved per-shard manifest')

## A100 production gate and checkpointing

The model forward computes loss against aligned target tensors. For 2048-token corpus blocks, `x=ids[:-1]`, `y=ids[1:]`, therefore **2047 supervised positions** per block. This is deliberately different from the synthetic v1 2049-token windows. Optimizer/scheduler/checkpoint code is adapted from v1; update-indexed sampling removes sampler-state ambiguity. Saving checkpoint to Drive can be slow. No run is started unless `PRODUCTION=True` **and** the licensing gate passes.

In [ ]:
MICROBATCH=8
ACCUM=4
SEQ=2047
TOKENS_PER_UPDATE=MICROBATCH*ACCUM*SEQ
LR=3e-4; MIN_LR=3e-5; WARMUP=2000; WEIGHT_DECAY=0.1; GRAD_CLIP=1.0
TARGET_TOKENS=20*sum(p.numel() for p in model.parameters())
TARGET_UPDATES=math.ceil(TARGET_TOKENS/TOKENS_PER_UPDATE)
RUN_DIR=PROJECT/'pretraining_05b_ssd_v2_1'
RUN_ID='maia_05b_ssd_v2_1'
# Provide the reviewed SHA-256 fingerprint of the exact tokenizer used for corpus v8.
# This is NOT inferred from the vocabulary size.
TOKENIZER_SHA256 = None
IDENTITY={'run_id':RUN_ID,'model_config':asdict(cfg),'params':sum(p.numel() for p in model.parameters()),
          'microbatch':MICROBATCH,'accum':ACCUM,'seq':SEQ,'seed':SEED,
          'source_weights':SOURCE_WEIGHTS,'approval_sha256':approval_sha,'tokenizer_sha256':TOKENIZER_SHA256,
          'dataset_blocks':info['total_blocks'],'target_updates':TARGET_UPDATES}
print('Parameters:',IDENTITY['params'],'updates:',TARGET_UPDATES,'tokens/update:',TOKENS_PER_UPDATE)
print('PRODUCTION:',PRODUCTION,'LICENSE_GATE:',LICENSE_GATE)
assert not PRODUCTION or LICENSE_GATE, 'FAIL CLOSED: approved shard eligibility is required'
assert not PRODUCTION or (isinstance(TOKENIZER_SHA256,str) and len(TOKENIZER_SHA256)==64 and all(x in '0123456789abcdef' for x in TOKENIZER_SHA256.lower())), 'FAIL CLOSED: reviewed tokenizer SHA-256 required'

In [ ]:
def lr_at(step):
    if step<WARMUP:return LR*(step+1)/WARMUP
    progress=min(1.,max(0.,(step-WARMUP)/max(1,TARGET_UPDATES-WARMUP)))
    return MIN_LR+0.5*(LR-MIN_LR)*(1+math.cos(math.pi*progress))

def save_checkpoint(step,model,optimizer):
    RUN_DIR.mkdir(parents=True,exist_ok=True)
    dest=RUN_DIR/f'checkpoint_{step:07d}.pt';tmp=dest.with_suffix('.pt.tmp')
    state={'identity':IDENTITY,'step':step,'model':model.state_dict(),
           'optimizer':optimizer.state_dict(),'python_rng':random.getstate(),
           'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),
           'cuda_rng':torch.cuda.get_rng_state_all(),
           'saved_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
    torch.save(state,tmp)
    os.replace(tmp,dest)
    print('Checkpoint saved:',dest,'GiB:',round(dest.stat().st_size/2**30,2))

def load_latest(model,optimizer):
    files=sorted(RUN_DIR.glob('checkpoint_*.pt')) if RUN_DIR.exists() else []
    if not files:return 0
    path=files[-1]
    ck=torch.load(path,map_location='cpu',weights_only=False)
    assert ck['identity']==IDENTITY,'Checkpoint identity differs from data/license/config'
    model.load_state_dict(ck['model']);optimizer.load_state_dict(ck['optimizer'])
    for state in optimizer.state.values():
        for k,v in state.items():
            if torch.is_tensor(v):state[k]=v.to('cuda')
    random.setstate(ck['python_rng']);np.random.set_state(ck['numpy_rng'])
    torch.set_rng_state(ck['torch_rng']);torch.cuda.set_rng_state_all(ck['cuda_rng'])
    step=int(ck['step']);del ck;gc.collect()
    print('Resuming from',path,'step',step)
    return step

## Controlled A100 smoke test (opt-in)

**Default execution does nothing.** Once rights have been reviewed and the approval file independently created, set `PRODUCTION=True` in the first cell and rerun. Initially use `UPDATES_TO_RUN=2`, `CHECKPOINT_EVERY=1`. Validate resumed results before extending the run. Checkpointing is at update boundaries; a Colab interruption mid-update replays that update deterministically.

In [ ]:
UPDATES_TO_RUN=2
CHECKPOINT_EVERY=1
if not PRODUCTION:
    print('SAFE STOP: production training disabled by default')
else:
    assert LICENSE_GATE and corpus is not None
    assert torch.cuda.is_available(), 'Select A100 80GB runtime'
    assert torch.cuda.get_device_properties(0).total_memory>=70*2**30
    assert torch.cuda.is_bf16_supported()
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
    model=model.to('cuda')
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,betas=(0.9,0.95),eps=1e-8,weight_decay=WEIGHT_DECAY)
    start=load_latest(model,optimizer)
    model.train()
    for step in range(start,min(start+UPDATES_TO_RUN,TARGET_UPDATES)):
        t0=time.perf_counter();optimizer.zero_grad(set_to_none=True)
        for group in optimizer.param_groups:group['lr']=lr_at(step)
        loss_sum=0.
        for micro in range(ACCUM):
            x,y=corpus.batch(step,micro,MICROBATCH)
            x=x.to('cuda',non_blocking=True);y=y.to('cuda',non_blocking=True)
            with torch.autocast('cuda',dtype=torch.bfloat16):
                _,loss=model(x,y)
            (loss/ACCUM).backward()
            loss_sum+=float(loss.detach())/ACCUM
            del x,y,loss
        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
        assert math.isfinite(loss_sum) and math.isfinite(float(norm))
        optimizer.step();torch.cuda.synchronize()
        elapsed=time.perf_counter()-t0
        print(f'update={step+1} loss={loss_sum:.4f} grad_norm={float(norm):.3f} '
              f'lr={lr_at(step):.3g} tokens/s={TOKENS_PER_UPDATE/elapsed:.0f}',flush=True)
        if (step+1)%CHECKPOINT_EVERY==0 or step+1==TARGET_UPDATES:
            save_checkpoint(step+1,model,optimizer)

## Verificação final sem iniciar treinamento

Esta célula confirma que a consolidação está pronta, mas **não aprova licenças** nem ativa a A100.

In [ ]:
assert MANIFEST.is_file() and info['total_blocks'] == 3607119
assert len(entries) == 18
assert PRODUCTION is False or (LICENSE_GATE and TOKENIZER_SHA256)
print('PASS: SSD-FIRST DATASET READY')
print('Shards:', info['total_shards'], '| blocks:', info['total_blocks'])
print('LICENSE_GATE:', LICENSE_GATE, '| PRODUCTION:', PRODUCTION)
print('Approved train blocks:', sum(counts.values()))

## Limitations and validation checklist

- **Licensing:** requires externally reviewed per-shard eligibility records. Pending Wikimedia rights are not approved by this notebook. Per-shard approvals may be insufficient if eligibility differs among documents within a shard; in that case regenerate eligible-only shards and indices, then re-audit.
- **Tokenizer:** this notebook validates token ID range, not the tokenizer fingerprint. Production requires comparison with the frozen corpus tokenizer artifact before setting `PRODUCTION=True`.
- **Numerical reproducibility:** stateless sampling is deterministic, but bitwise GPU resume still requires an uninterrupted-versus-resumed test on the exact A100/PyTorch build. BF16 SDPA kernels may not be bitwise reproducible across versions.
- **Data mixture:** SOURCE_WEIGHTS are provisional; define scientifically justified language targets and sampling with replacement explicitly.
- **Performance:** batch creation occurs synchronously on CPU in this reference implementation. Profile and prefetch before long production runs.
- **Validation:** add a fixed, independently approved validation set and periodic loss/perplexity monitoring before extended pretraining.
- **Checkpoint durability:** checkpoints are saved atomically on the Drive mount; verify checksum/read-back for production-grade recovery.

## Ensaio v2.2 — retomada determinística em CPU (dados sintéticos)

Este teste **não** usa o corpus v8, não altera `LICENSE_GATE` e não executa produção. Usa uma instância pequena da **mesma classe MaiaGPT** para comparar 6 updates contínuos contra 3 + checkpoint + 3. Salva checkpoint em diretório temporário local, nunca no Drive. Verifica igualdade exata dos pesos, AdamW, perdas, batches e estados RNG. O teste não substitui a futura verificação na A100 BF16.


In [ ]:
import copy, tempfile, pickle
from pathlib import Path

# Teste de CPU com parâmetros reduzidos, mantendo a implementação MaiaGPT.
_old_threads = torch.get_num_threads()
torch.set_num_threads(1)
TEST_CFG = Config(vocab_size=257, block_size=33, n_layer=2,
                  n_head=4, n_embd=64, dropout=0.0, bias=True)
TEST_SEED = 20261107
TEST_UPDATES = 6
TEST_SPLIT = 3
TEST_BATCH = 2
TEST_SEQ = 32
TEST_ACCUM = 2

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def synthetic_batch(step, micro):
    # Fonte independente do estado global: endereço (step, micro) determina os dados.
    digest = hashlib.sha256(f'maia-v22:{TEST_SEED}:{step}:{micro}'.encode()).digest()
    rng = np.random.default_rng(int.from_bytes(digest[:8], 'little'))
    raw = rng.integers(0, TEST_CFG.vocab_size,
                       size=(TEST_BATCH, TEST_SEQ + 1), dtype=np.int64)
    return (torch.from_numpy(raw[:, :-1].copy()),
            torch.from_numpy(raw[:, 1:].copy()),
            hashlib.sha256(raw.tobytes()).hexdigest())

def new_test_run():
    seed_all(TEST_SEED)
    m = MaiaGPT(TEST_CFG).cpu().train()
    o = torch.optim.AdamW(m.parameters(), lr=1e-3, betas=(0.9, 0.95),
                          weight_decay=0.1, foreach=False)
    return m, o

def test_updates(m, o, start, stop, log):
    for step in range(start, stop):
        o.zero_grad(set_to_none=True)
        hashes = []
        total_loss = 0.0
        for micro in range(TEST_ACCUM):
            x, y, batch_hash = synthetic_batch(step, micro)
            hashes.append(batch_hash)
            _, loss = m(x, y)
            (loss / TEST_ACCUM).backward()
            total_loss += float(loss.detach()) / TEST_ACCUM
        norm = torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        assert math.isfinite(total_loss) and math.isfinite(float(norm))
        o.step()
        log.append({'step': step + 1, 'loss': total_loss,
                    'batch_hashes': hashes})

def snapshot(m, o):
    # Captura profunda para impedir aliasing com os tensores mutáveis do modelo.
    return {'model': {k: v.detach().clone() for k, v in m.state_dict().items()},
            'optimizer': copy.deepcopy(o.state_dict()),
            'python_rng': random.getstate(),
            'numpy_rng': np.random.get_state(),
            'torch_rng': torch.get_rng_state().clone()}

def save_test_checkpoint(path, m, o, step):
    state = snapshot(m, o)
    state.update({'step': step, 'test_cfg': asdict(TEST_CFG),
                  'test_seed': TEST_SEED, 'format': 'maia-v22-cpu-determinism'})
    tmp = path.with_suffix('.tmp')
    torch.save(state, tmp)
    os.replace(tmp, path)

def load_test_checkpoint(path):
    state = torch.load(path, map_location='cpu', weights_only=False)
    assert state['format'] == 'maia-v22-cpu-determinism'
    assert state['test_cfg'] == asdict(TEST_CFG)
    assert state['test_seed'] == TEST_SEED
    m, o = new_test_run()
    m.load_state_dict(state['model'])
    o.load_state_dict(state['optimizer'])
    random.setstate(state['python_rng'])
    np.random.set_state(state['numpy_rng'])
    torch.set_rng_state(state['torch_rng'])
    return m, o, int(state['step'])

def exact_equal(a, b):
    if torch.is_tensor(a) and torch.is_tensor(b):
        return a.dtype == b.dtype and a.shape == b.shape and torch.equal(a, b)
    if isinstance(a, np.ndarray) and isinstance(b, np.ndarray):
        return np.array_equal(a, b)
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(exact_equal(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(exact_equal(x, y) for x, y in zip(a, b))
    return a == b

try:
    continuous_log = []
    m1, o1 = new_test_run()
    test_updates(m1, o1, 0, TEST_UPDATES, continuous_log)
    continuous_state = snapshot(m1, o1)
    del m1, o1

    resumed_log = []
    m2, o2 = new_test_run()
    test_updates(m2, o2, 0, TEST_SPLIT, resumed_log)
    with tempfile.TemporaryDirectory(prefix='maia_v22_test_') as tmpdir:
        checkpoint = Path(tmpdir) / 'resume.pt'
        save_test_checkpoint(checkpoint, m2, o2, TEST_SPLIT)
        del m2, o2
        m2, o2, restored_step = load_test_checkpoint(checkpoint)
        assert restored_step == TEST_SPLIT
        test_updates(m2, o2, restored_step, TEST_UPDATES, resumed_log)
    resumed_state = snapshot(m2, o2)
    del m2, o2

    checks = {
        'batch_sequence': [r['batch_hashes'] for r in continuous_log] ==
                          [r['batch_hashes'] for r in resumed_log],
        'losses_exact': [r['loss'] for r in continuous_log] ==
                        [r['loss'] for r in resumed_log],
        'model_exact': exact_equal(continuous_state['model'], resumed_state['model']),
        'adamw_exact': exact_equal(continuous_state['optimizer'], resumed_state['optimizer']),
        'python_rng_exact': exact_equal(continuous_state['python_rng'], resumed_state['python_rng']),
        'numpy_rng_exact': exact_equal(continuous_state['numpy_rng'], resumed_state['numpy_rng']),
        'torch_rng_exact': exact_equal(continuous_state['torch_rng'], resumed_state['torch_rng']),
    }
    for name, passed in checks.items():
        print(('PASS' if passed else 'FAIL') + ': ' + name)
    assert all(checks.values()), 'Retomada NÃO determinística; investigar antes de treinar'
    print('PASS: DETERMINISTIC CPU RESUME — 6 vs 3+3 updates')
    print('LICENSE_GATE não foi modificado; nenhum dado real foi usado.')
finally:
    torch.set_num_threads(_old_threads)


### Interpretação e limites

- `PASS` comprova igualdade exata **neste ensaio CPU FP32 com modelo reduzido**, não na A100 BF16.
- O teste usa a mesma implementação de `MaiaGPT`, mas o caminho de checkpoint de produção e o sampler real ainda precisam de ensaios separados.
- O teste não aprova direitos sobre o corpus v8 e não ativa `PRODUCTION`.
- Próximo teste: execução curta em A100 com dados sintéticos e checkpoint de produção em ambiente controlado, antes de qualquer treino longo.
